In [ ]:
import torch
import torch.onnx as to
# generic model
class AddSigmoid(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return self.sigmoid(x + y)

example_inputs = (torch.ones(1,1,1,1),torch.ones(1,1,1,1))

model = AddSigmoid()
model = model.eval()
exported_model= torch.export.export(model, example_inputs)
graph_module = exported_model.module(check_guards=False)

_ = graph_module.print_readable()

# Create a new exported program using the graph_module
exported_graph = torch.export.export(graph_module, example_inputs) # will highlight why we need this later 

In [ ]:
# cell for making the .tosa dump for the model FP variant

from pathlib import Path
from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.arm.tosa import TosaSpecification
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.util._factory import create_partitioner


BASE_DUMP = Path("Tosa_dump")

def dump_tosa(ep, profile_str: str, label: str):
    BASE_DUMP.mkdir(parents=True, exist_ok=True)

    tosa_spec = TosaSpecification.create_from_string(profile_str)
    compile_spec = TosaCompileSpec(tosa_spec)

    # Force ALL artifacts into the same folder
    compile_spec.dump_intermediate_artifacts_to(str(BASE_DUMP))

    partitioner = create_partitioner(compile_spec)

    _ = to_edge_transform_and_lower(
        ep,
        partitioner=[partitioner],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    )

    tosa_files = list(BASE_DUMP.rglob("*.tosa"))

    print(f"\n{label}")
    print(f"  Profile: {profile_str}")
    print(f"  Dump dir: {BASE_DUMP.resolve()}")
    print(f"  Total .tosa files so far: {len(tosa_files)}")

dump_tosa(exported_model, "TOSA-1.0+FP",
    "AddSigmoidModel (float)")



Copy the below command into the terminal to run the model explorer: 

    model-explorer \
    --extensions=pte_adapter_model_explorer

In [ ]:
from executorch.backends.arm.vgf import VgfCompileSpec, VgfPartitioner
from executorch.exir import (
    EdgeCompileConfig,
    ExecutorchBackendConfig,
    to_edge_transform_and_lower,
)
from executorch.extension.export_util.utils import save_pte_program
import os
import torch

os.makedirs("executorch_model", exist_ok=True)


# VGF lowering
compile_spec = VgfCompileSpec()    
compile_spec.dump_intermediate_artifacts_to("executorch_model")   # default FP compile spec
partitioner = VgfPartitioner(compile_spec)

edge_pm = to_edge_transform_and_lower(
    exported_model,
    partitioner=[partitioner],
    compile_config=EdgeCompileConfig(_check_ir_validity=False),
)

et_pm = edge_pm.to_executorch(
    config=ExecutorchBackendConfig(extract_delegate_segments=False)
)

# Save .pte
cwd_dir = os.getcwd()
pte_base_name = "AS_vgf"
out_name = pte_base_name + ".pte"
save_pte_program(et_pm, out_name)
print("Wrote:", os.path.abspath(out_name))
pte_path = os.path.join(cwd_dir, out_name)

graph_module = exported_model.module(check_guards=False)
_ = graph_module.print_readable() # for visibility 

In [ ]:
%%bash
# Ensure the vulkan environment variables and MLSDK components are available on $PATH
source repo/executorch/examples/arm/arm-scratch/setup_path.sh
cd repo/executorch/examples/arm

# Compiled programs will appear in the executorch/cmake-out directory we create here.
# Build example executor runner application to examples/arm/vgf_minimal_example
cmake \
  -DCMAKE_INSTALL_PREFIX=cmake-out \
  -DCMAKE_BUILD_TYPE=Debug \
  -DEXECUTORCH_BUILD_EXTENSION_DATA_LOADER=ON \
  -DEXECUTORCH_BUILD_EXTENSION_MODULE=ON \
  -DEXECUTORCH_BUILD_EXTENSION_NAMED_DATA_MAP=ON \
  -DEXECUTORCH_BUILD_EXTENSION_FLAT_TENSOR=ON \
  -DEXECUTORCH_BUILD_EXTENSION_TENSOR=ON \
  -DEXECUTORCH_BUILD_KERNELS_QUANTIZED=ON \
  -DEXECUTORCH_BUILD_XNNPACK=OFF \
  -DEXECUTORCH_BUILD_VULKAN=ON \
  -DEXECUTORCH_BUILD_VGF=ON \
  -DEXECUTORCH_ENABLE_LOGGING=ON \
  -DPYTHON_EXECUTABLE=python \
  -B../../cmake-out-vkml ../..

cmake --build ../../cmake-out-vkml --target executor_runner

In [ ]:
import subprocess

cwd_dir = os.getcwd()


# Setup paths
et_dir = os.path.join(cwd_dir)

script_dir = os.path.join(et_dir, "repo", "executorch" ,"backends", "arm", "scripts")
et_dir = os.path.join(et_dir, "repo" , "executorch")

args = f"--model={pte_path}"
subprocess.run(os.path.join(script_dir, "run_vkml.sh") + " " + args, shell=True, cwd=et_dir)

In [ ]:
# code to convert the .tosa of the qunatised model to a .vgf
import pathlib
tosa_path = pathlib.Path("./Tosa_dump/output_tag0_TOSA-1.0+FP.tosa")
vgf_path = pathlib.Path("./executorch_model/model.vgf")

subprocess.run(
    [
        "model_converter",
        "--input",str(tosa_path),
        "--output",str(vgf_path),
    ],
    check=True,
)

In [ ]:
import subprocess

cwd_dir = os.getcwd()
vgf_path = os.path.join(cwd_dir, "model.vgf")  # absolute path

# Setup paths
et_dir = os.path.join(cwd_dir)

script_dir = os.path.join(et_dir, "repo", "executorch" ,"backends", "arm", "scripts")
et_dir = os.path.join(et_dir, "repo" , "executorch")

args = f"--model={vgf_path}"
subprocess.run(os.path.join(script_dir, "run_vkml.sh") + " " + args, shell=True, cwd=et_dir)